In [0]:
# ============================================
# CityFlow — 02_silver_transformation
# ============================================
import urllib.parse

# Secure credentials
ACCESS_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-access-key")
SECRET_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-secret-key")
encoded_secret = urllib.parse.quote(SECRET_KEY, safe="")

S3_BUCKET = "cityflow-taxi-data"
BRONZE_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/bronze"
SILVER_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/silver"

# Read from Bronze
df_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_PATH}/taxi/")

print(f"Bronze records: {df_bronze.count()}")

Bronze records: 10906858


In [0]:
from pyspark.sql.functions import col, hour, dayofweek

df_silver = df_bronze \
    .dropna(subset=["tpep_pickup_datetime", "fare_amount", "pickup_longitude"]) \
    .filter(col("fare_amount").between(1, 500)) \
    .filter(col("trip_distance") > 0) \
    .filter(col("passenger_count").between(1, 6)) \
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day", dayofweek("tpep_pickup_datetime")) \
    .withColumn("trip_date", col("tpep_pickup_datetime").cast("date"))

# Quality check
total = df_silver.count()
print(f"Clean records: {total}")

# Save  Silver
df_silver.write.format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/taxi/")

print("Silver layer saved!")

Clean records: 10836815
Silver layer saved!
